# 00 · Data Understanding & Feature Selection
**AIE314/AIE315 · ชุดข้อมูลที่ 4 ปริมาณการเดินทางและผู้โดยสาร** · ผู้รับผิดชอบ: สมาชิก A

เป้าหมาย: ทำความเข้าใจโครงสร้างไฟล์ต้นฉบับ ความหมายของแต่ละแถว/คอลัมน์ และหลักฐานของปัญหาคุณภาพข้อมูล
ก่อนเข้าสู่ `01_cleaning.ipynb` (notebook นี้ **ไม่แก้ไขข้อมูล** แค่สำรวจ)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
prep.setup_thai_font()

'Sukhumvit Set'

## 1. โหลดไฟล์ต้นฉบับ (อ่านเป็นข้อความทั้งหมด)

In [2]:
raw_text = {}
for year_be, path in prep.RAW_FILES.items():
    raw_text[year_be] = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    print(year_be, path.name, raw_text[year_be].shape)
raw_text[2568].head()

2568 4_passengers68.csv (69440, 8)
2569 4_passengers69.csv (3010, 8)


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


## 2. Data Dictionary (ไฟล์ต้นฉบับ)
| คอลัมน์ | ชนิดในไฟล์ | ความหมาย | ตัวอย่าง |
|---|---|---|---|
| รูปแบบการเดินทาง | ข้อความ | ทางถนน / ทางน้ำ / ทางราง / ทางอากาศ | ทางถนน |
| วัตถุประสงค์ | ข้อความ | ระหว่างจังหวัด / ภายในจังหวัด-กรุงเทพ / ระหว่างประเทศ | การเดินทางระหว่างจังหวัด |
| สาธารณะ/ส่วนบุคคล | ข้อความ | ประเภทการเดินทาง | สาธารณะ |
| หน่วยงาน | ข้อความ | หน่วยงานที่รายงานข้อมูล | บขส. |
| ยานพาหนะ/ท่า | ข้อความ | ชื่อบริการ/ท่า (บางรายการมีทิศ ขาเข้า/ขาออกประเทศ ต่อท้าย) | รถ บขส. และ รถร่วม |
| วันที่ | ข้อความ `dd/mm/yyyy` (ค.ศ.) | วันที่ของข้อมูล | 01/01/2025 |
| หน่วย | ข้อความ | คน หรือ คัน | คน |
| ปริมาณ | ข้อความมี comma | จำนวนผู้โดยสาร/ยานพาหนะต่อวัน | "127,551" |

**1 แถว = ปริมาณของ 1 บริการ ใน 1 วัน** (long format)

## 3. แถวว่าง

In [3]:
for year_be, df in raw_text.items():
    blank = (df.apply(lambda c: c.str.strip()) == "").all(axis=1)
    print(f"{year_be}: rows={len(df):,}  blank rows={blank.sum():,}  data rows={(~blank).sum():,}")

2568: rows=69,440  blank rows=53,744  data rows=15,696
2569: rows=3,010  blank rows=0  data rows=3,010


## 4. ค่าที่เป็นไปได้ในแต่ละคอลัมน์ (หลังตัดแถวว่าง)

In [4]:
raw = {y: prep.load_raw(p) for y, p in prep.RAW_FILES.items()}
for col in ["รูปแบบการเดินทาง", "วัตถุประสงค์", "สาธารณะ/ส่วนบุคคล", "หน่วยงาน", "หน่วย"]:
    print(col, "->", sorted(raw[2568][col].unique()))
print("จำนวนบริการ:", {y: d["ยานพาหนะ/ท่า"].nunique() for y, d in raw.items()})
print("บริการที่มีเพียงปีเดียว:", set(raw[2568]["ยานพาหนะ/ท่า"]) ^ set(raw[2569]["ยานพาหนะ/ท่า"]))

รูปแบบการเดินทาง -> ['ทางถนน', 'ทางน้ำ', 'ทางราง', 'ทางอากาศ']
วัตถุประสงค์ -> ['การเดินทางภายในจังหวัด/กรุงเทพ', 'การเดินทางระหว่างจังหวัด', 'การเดินทางระหว่างประเทศ']
สาธารณะ/ส่วนบุคคล -> ['สาธารณะ', 'ส่วนบุคคล']
หน่วยงาน -> ['กทพ.', 'กทม.', 'กพท.', 'ขบ.', 'ขสมก.', 'จท.', 'ทย.', 'ทล.', 'ทอท.', 'บขส.', 'รฟท.', 'รฟฟท.', 'รฟม.']
หน่วย -> ['คน', 'คัน']
จำนวนบริการ: {2568: 43, 2569: 43}
บริการที่มีเพียงปีเดียว: set()


## 5. ช่วงวันที่และความครบถ้วนของวัน

In [5]:
for y, d in raw.items():
    dates = pd.to_datetime(d["วันที่"], format="%d/%m/%Y")
    full = pd.date_range(dates.min(), dates.max())
    print(f"{y}: {dates.min().date()} -> {dates.max().date()}  unique days={dates.nunique()}  missing days={len(full.difference(dates))}")

2568: 2025-01-01 -> 2025-12-31  unique days=365  missing days=0
2569: 2026-01-01 -> 2026-03-11  unique days=70  missing days=0


## 6. สรุปต่อบริการ (หลักฐานของค่าว่าง ค่าศูนย์ และค่าซ้ำ)

In [6]:
std = pd.concat([prep.to_standard(d, y) for y, d in raw.items()], ignore_index=True)
summary = (std.groupby(["year_be", "purpose", "mode", "agency", "service", "unit"])
              .agg(n_rows=("value", "size"), n_missing=("value", lambda s: s.isna().sum()),
                   n_zero=("value", lambda s: (s == 0).sum()), mean=("value", "mean"),
                   min=("value", "min"), max=("value", "max"))
              .reset_index())
summary["mean"] = summary["mean"].round(0)
summary[(summary["n_missing"] > 0) | (summary["n_zero"] > 0) | (summary["n_rows"] != summary.groupby("year_be")["n_rows"].transform("median"))]

,year_be,purpose,mode,agency,service,unit,n_rows,n_missing,n_zero,mean,min,max
0,2568,intercity,air,กพท.,ท่าอากาศยานสมุย,คน,365,0,2,3468.0,0.0,5682.0
1,2568,intercity,air,กพท.,ท่าอากาศยานอู่ตะเภา,คน,365,0,2,215.0,0.0,467.0
6,2568,intercity,rail,รฟท.,รถไฟ,คน,365,1,2,74689.0,0.0,106219.0
9,2568,intercity,road,ขบ.,รถหมวด 3,คน,365,0,93,2822.0,0.0,8218.0
10,2568,intercity,road,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),คัน,365,143,0,974738.0,849407.0,1136940.0
11,2568,intercity,road,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),คัน,365,143,0,830150.0,744794.0,984400.0
12,2568,intercity,road,บขส.,รถ บขส. และ รถร่วม,คน,365,2,0,91832.0,63186.0,908667.0
13,2568,international,air,ทย.,ท่าอากาศยานภูมิภาค ขาออกประเทศ,คน,366,0,0,1089.0,94.0,2252.0
14,2568,international,air,ทย.,ท่าอากาศยานภูมิภาค ขาเข้าประเทศ,คน,365,0,1,1047.0,0.0,2631.0
21,2568,international,rail,รฟท.,รถไฟ ขาออกประเทศ,คน,365,0,5,319.0,0.0,761.0


## 7. ข้อมูลซ้ำ

In [7]:
for y in raw:
    d = std[std["year_be"] == y]
    dup = d[d.duplicated(["service", "date"], keep=False)]
    print(y, "duplicate (service, date) rows:", len(dup))
    display(dup[["service", "date", "value"]])

2568 duplicate (service, date) rows: 2


,service,date,value
12641,ท่าอากาศยานภูมิภาค ขาออกประเทศ,2025-10-21,1257.0
12642,ท่าอากาศยานภูมิภาค ขาออกประเทศ,2025-10-21,1257.0


2569 duplicate (service, date) rows: 0


,service,date,value


## 8. รายการที่ซ้อนกัน: “รถยนต์ทุกประเภท” ควร ≥ “รถยนต์เฉพาะ 4 ล้อ”

In [8]:
for agency in ["ทล.", "กทพ."]:
    p = std[std["agency"] == agency].pivot_table(index=["year_be", "date"], columns="service", values="value")
    all_col = [c for c in p.columns if "ทุกประเภท" in c][0]
    four_col = [c for c in p.columns if "4 ล้อ" in c][0]
    bad = p[p[all_col] < p[four_col]]
    print(f"{agency}: days where all-types < 4-wheel = {len(bad)} of {p.dropna().shape[0]} comparable days")
    display(bad.head(10))

ทล.: days where all-types < 4-wheel = 0 of 227 comparable days


,service,รถยนต์ทุกประเภท (10 จุดสำรวจ),รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ)
year_be,date,,


กทพ.: days where all-types < 4-wheel = 5 of 435 comparable days


service             รถยนต์ทุกประเภท (ทางด่วน)  รถยนต์เฉพาะ 4 ล้อ (ทางด่วน)
year_be date                                                              
2568    2025-01-01                   979129.0                    1364992.0
        2025-01-02                   947189.0                    1286160.0
        2025-01-30                  1517654.0                    1641282.0
        2025-01-31                  1399387.0                    1689391.0
        2025-10-10                  2301447.0                    2373044.0

## 9. Feature Selection (สรุปสำหรับรายงาน)
| Feature ต้นฉบับ | ใช้ในข้อ | เหตุผล |
|---|---|---|
| วันที่ | 1, 2, 3 | สร้างวันในสัปดาห์ / เดือน / ช่วงเทียบข้ามปี |
| ยานพาหนะ/ท่า | 1, 2, 3 | หน่วยการวิเคราะห์ (บริการ) และแหล่งของทิศขาเข้า/ขาออก |
| วัตถุประสงค์ | 1, 3 | แยกการเดินทางในเมือง / ระหว่างจังหวัด / ระหว่างประเทศ |
| รูปแบบการเดินทาง | 1, 3 | จัดกลุ่มตามโหมด (ข้อ 3 ต้องจำแนกตามรูปแบบการเดินทาง) |
| หน่วย | 1, 2, 3 | ป้องกันการรวม “คน” กับ “คัน” |
| ปริมาณ | 1, 2, 3 | ตัวแปรเป้าหมาย |
| หน่วยงาน | (อธิบาย) | ใช้ตรวจรายการซ้อนกัน ไม่ใช้เป็นแกนวิเคราะห์ |
| สาธารณะ/ส่วนบุคคล | (อธิบาย) | ส่วนบุคคลมีเฉพาะรายการหน่วย “คัน” จึงซ้ำกับคอลัมน์หน่วย |